# A06. Galton Board 시뮬레이션

**학습 목표**
- Galton Board(골턴 보드)의 물리적 구조를 이해하고 시뮬레이션으로 구현한다
- 1개 동전 10번 던지기 실험을 1,490회 수행하여 이항분포를 확인한다
- 10개 동전 한꺼번에 던지기(H=+1, T=-1, 합에서 5 차감)와의 동치성을 증명한다
- $n$이 커지면 정규분포에 수렴함을 시각적으로 확인한다

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import binom, bernoulli, norm, uniform, expon, pareto
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
sns.set_style('whitegrid')
sns.set_palette('Set2')

np.random.seed(42)
print('라이브러리 로드 완료')

---
## 1. Galton Board 개념

### Galton Board(골턴 보드)란?

1889년 **Francis Galton**이 고안한 물리적 장치로, 구슬이 못(핀) 배열을 통과하면서 **이항분포 → 정규분포**를 시각적으로 보여준다.

```
        ●           ← 구슬 투입
       / \
      ●   ●         ← 1단계: 좌/우 50%
     / \ / \
    ●   ●   ●       ← 2단계: 좌/우 50%
   / \ / \ / \
  ●   ●   ●   ●     ← 3단계
  ...
 [__|__|__|__|__]    ← 칸(bin)에 쌓임
```

### 핵심 원리

- 각 핀에서 **좌 또는 우**로 갈 확률이 각각 $0.5$ → **베르누이 시행**
- $n$단계를 통과하면 오른쪽으로 간 횟수 $k \sim B(n, 0.5)$ → **이항분포**
- 구슬이 충분히 많으면 칸에 쌓인 모양이 **종형 곡선(정규분포)**

> **베르누이 → 이항 → 정규분포**를 하나의 물리적 장치로 연결!

---
## 2. 시뮬레이션 1: 1개 동전 10번 던지기 × 1,490회

In [ ]:
# === 시뮬레이션 1: 1개 동전 10번 던지기를 1490회 반복 ===
n_flips = 10       # 한 번에 던지는 횟수
n_experiments = 1490  # 실험 반복 횟수

# 각 실험에서 앞면(Head) 개수를 카운트
# 방법: 베르누이(0.5)를 10번 시행 → 합 = 앞면 개수
results_sim1 = np.array([
    np.sum(bernoulli.rvs(0.5, size=n_flips)) 
    for _ in range(n_experiments)
])

print(f'=== 시뮬레이션 1: 동전 1개 × {n_flips}번 던지기 × {n_experiments}회 ===')
print(f'결과 앞면 개수 (처음 20개): {results_sim1[:20]}')
print(f'평균: {np.mean(results_sim1):.4f}  (이론: {n_flips * 0.5})')
print(f'분산: {np.var(results_sim1):.4f}  (이론: {n_flips * 0.5 * 0.5})')

In [ ]:
# === 시뮬레이션 1 시각화: 히스토그램 + 이론 이항분포 오버레이 ===
k_values = np.arange(0, n_flips + 1)
theoretical_pmf = binom.pmf(k_values, n_flips, 0.5)
theoretical_counts = theoretical_pmf * n_experiments  # 기대 빈도

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# (1) 히스토그램 vs 이론 PMF (확률)
axes[0].hist(results_sim1, bins=np.arange(-0.5, n_flips + 1.5, 1),
             density=True, color=sns.color_palette('Set2')[0],
             edgecolor='black', alpha=0.7, label=f'시뮬레이션 ({n_experiments}회)')
axes[0].plot(k_values, theoretical_pmf, 'ro-', markersize=8, linewidth=2.5,
             label=f'이론 B({n_flips}, 0.5)', zorder=5)
axes[0].set_xlabel('앞면 개수 k', fontsize=13)
axes[0].set_ylabel('상대 빈도 / 확률', fontsize=13)
axes[0].set_title(f'Galton Board 시뮬레이션 1\n(동전 1개 × 10번 × {n_experiments}회)',
                   fontsize=14, fontweight='bold')
axes[0].set_xticks(k_values)
axes[0].legend(fontsize=11)

# (2) 빈도 비교 (실제 vs 기대)
actual_counts = np.array([np.sum(results_sim1 == k) for k in k_values])
width = 0.35
axes[1].bar(k_values - width/2, actual_counts, width, color=sns.color_palette('Set2')[0],
            edgecolor='black', alpha=0.8, label='실측 빈도')
axes[1].bar(k_values + width/2, theoretical_counts, width, color=sns.color_palette('Set2')[1],
            edgecolor='black', alpha=0.8, label='이론 빈도')
axes[1].set_xlabel('앞면 개수 k', fontsize=13)
axes[1].set_ylabel('빈도 (횟수)', fontsize=13)
axes[1].set_title('실측 빈도 vs 이론 빈도 비교', fontsize=14, fontweight='bold')
axes[1].set_xticks(k_values)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

# 빈도 비교 테이블
comparison_df = pd.DataFrame({
    '앞면 개수 k': k_values,
    '실측 빈도': actual_counts,
    '이론 빈도': np.round(theoretical_counts, 1),
    '차이': np.round(actual_counts - theoretical_counts, 1)
})
print('\n=== 빈도 비교 테이블 ===')
print(comparison_df.to_string(index=False))

### 해석

- 1,490회 실험 결과가 이론적 이항분포 $B(10, 0.5)$와 매우 잘 일치한다.
- **가장 빈번한 결과**: 앞면 5개 (약 24.6%). 이는 이항분포의 최빈값(mode)과 일치.
- 실측 빈도와 이론 빈도의 차이가 대부분 10 이내로, 시뮬레이션의 신뢰성을 확인할 수 있다.
- 이것이 바로 Galton Board에서 구슬 1,490개를 떨어뜨렸을 때 각 칸에 쌓이는 모습이다.

---
## 3. 시뮬레이션 2: 10개 동전 한꺼번에 × 149회

### 변환 규칙
- 앞면(H) = **+1**, 뒷면(T) = **-1**
- 10개 동전의 합 계산 후 **5를 차감**

### 수학적 동치성

10개 동전 중 앞면 개수를 $k$라 하면:
- 합 = $k \times (+1) + (10-k) \times (-1) = 2k - 10$
- 5를 차감하면: $2k - 10 - 5 = 2k - 15$

이 값의 분포는 $k \sim B(10, 0.5)$에서 선형 변환된 것이므로, 기본적으로 **동일한 이항분포**를 다른 스케일로 표현한 것이다.

In [ ]:
# === 시뮬레이션 2: 10개 동전 한꺼번에 × 149회 ===
n_coins = 10         # 한꺼번에 던지는 동전 수
n_experiments2 = 149  # 실험 횟수

results_sim2_raw = []      # 앞면 개수 (원래 스케일)
results_sim2_transformed = []  # H=+1, T=-1 합에서 5 차감

for _ in range(n_experiments2):
    # 10개 동전을 한번에 던짐 (1=앞면, 0=뒷면)
    coins = bernoulli.rvs(0.5, size=n_coins)
    
    # 앞면 개수
    heads_count = np.sum(coins)
    results_sim2_raw.append(heads_count)
    
    # H=+1, T=-1 변환 후 합에서 5 차감
    transformed = np.where(coins == 1, +1, -1)
    score = np.sum(transformed) - 5
    results_sim2_transformed.append(score)

results_sim2_raw = np.array(results_sim2_raw)
results_sim2_transformed = np.array(results_sim2_transformed)

print(f'=== 시뮬레이션 2: 동전 {n_coins}개 × {n_experiments2}회 ===')
print(f'앞면 개수 (처음 15개): {results_sim2_raw[:15]}')
print(f'변환 점수 (처음 15개): {results_sim2_transformed[:15]}')
print(f'\n앞면 개수 평균: {np.mean(results_sim2_raw):.4f}')
print(f'변환 점수 평균: {np.mean(results_sim2_transformed):.4f}')
print(f'변환 공식: score = 2k - 10 - 5 = 2k - 15 (k=앞면 개수)')

In [ ]:
# === 시뮬레이션 2 시각화 ===
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# (1) 앞면 개수 분포 (시뮬레이션 1과 비교)
k_vals = np.arange(0, n_coins + 1)
theory_pmf = binom.pmf(k_vals, n_coins, 0.5)

axes[0].hist(results_sim2_raw, bins=np.arange(-0.5, n_coins + 1.5, 1),
             density=True, color=sns.color_palette('Set2')[2],
             edgecolor='black', alpha=0.7, label=f'시뮬레이션 2 ({n_experiments2}회)')
axes[0].plot(k_vals, theory_pmf, 'ro-', markersize=8, linewidth=2.5,
             label=f'이론 B({n_coins}, 0.5)', zorder=5)
axes[0].set_xlabel('앞면 개수 k', fontsize=13)
axes[0].set_ylabel('상대 빈도 / 확률', fontsize=13)
axes[0].set_title(f'10개 동전 한꺼번에 × {n_experiments2}회\n(앞면 개수 분포)',
                   fontsize=14, fontweight='bold')
axes[0].set_xticks(k_vals)
axes[0].legend(fontsize=11)

# (2) 변환 점수 분포 (H:+1, T:-1, 합-5)
possible_scores = 2 * k_vals - 15  # 변환 공식
axes[1].hist(results_sim2_transformed, bins=20,
             density=True, color=sns.color_palette('Set2')[3],
             edgecolor='black', alpha=0.7, label='변환 점수 분포')
axes[1].axvline(np.mean(results_sim2_transformed), color='red', linestyle='--',
                linewidth=2, label=f'평균 = {np.mean(results_sim2_transformed):.2f}')
axes[1].set_xlabel('변환 점수 (2k - 15)', fontsize=13)
axes[1].set_ylabel('상대 빈도', fontsize=13)
axes[1].set_title(f'H=+1, T=-1 합에서 5 차감\n(변환된 스케일)',
                   fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

### 해석

- **왼쪽**: 10개 동전을 149회 던진 결과. 실험 횟수가 시뮬레이션 1(1490회)보다 적어 이론 PMF와의 차이가 다소 크지만, 전반적인 형태는 $B(10, 0.5)$를 따른다.
- **오른쪽**: $H=+1, T=-1$ 변환 후 합에서 5를 차감한 점수 분포. 이 역시 종형 분포를 보인다.
- 두 분포는 **스케일만 다를 뿐 본질적으로 동일한 분포**이다.

---
## 4. 두 시뮬레이션의 동치성 증명

In [ ]:
# === 두 시뮬레이션이 동일한 분포임을 증명 ===
# 동일한 시드로 두 방법을 비교
np.random.seed(123)
N = 10000

# 방법 1: 1개 동전 10번 던지기 → 앞면 개수
method1 = np.array([np.sum(bernoulli.rvs(0.5, size=10)) for _ in range(N)])

np.random.seed(123)
# 방법 2: 10개 동전 한꺼번에 → H=+1, T=-1 합
method2_sum = np.array([np.sum(np.where(bernoulli.rvs(0.5, size=10) == 1, 1, -1)) for _ in range(N)])

# 방법 2 결과를 앞면 개수로 역변환: sum = 2k - 10 → k = (sum + 10) / 2
method2_heads = (method2_sum + 10) / 2

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# (1) 방법 1: 앞면 개수
axes[0].hist(method1, bins=np.arange(-0.5, 11.5, 1), density=True,
             color=sns.color_palette('Set2')[0], edgecolor='black', alpha=0.7)
axes[0].set_title('방법 1: 1개 동전 × 10번\n(앞면 개수)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('앞면 개수', fontsize=12)
axes[0].set_ylabel('상대 빈도', fontsize=12)

# (2) 방법 2: +1/-1 합
axes[1].hist(method2_sum, bins=20, density=True,
             color=sns.color_palette('Set2')[2], edgecolor='black', alpha=0.7)
axes[1].set_title('방법 2: 10개 동전 한꺼번에\n(+1/-1 합)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('+1/-1 합', fontsize=12)
axes[1].set_ylabel('상대 빈도', fontsize=12)

# (3) 겹쳐서 비교 (앞면 개수 스케일)
axes[2].hist(method1, bins=np.arange(-0.5, 11.5, 1), density=True,
             color=sns.color_palette('Set2')[0], edgecolor='black', alpha=0.5, label='방법 1')
axes[2].hist(method2_heads, bins=np.arange(-0.5, 11.5, 1), density=True,
             color=sns.color_palette('Set2')[2], edgecolor='black', alpha=0.5, label='방법 2 (역변환)')
k_vals = np.arange(0, 11)
axes[2].plot(k_vals, binom.pmf(k_vals, 10, 0.5), 'ro-', markersize=8,
             linewidth=2, label='이론 B(10,0.5)', zorder=5)
axes[2].set_title('두 방법 비교\n(동일 분포 확인)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('앞면 개수', fontsize=12)
axes[2].set_ylabel('상대 빈도', fontsize=12)
axes[2].legend(fontsize=10)

plt.tight_layout()
plt.show()

# 통계적 비교
print('=== 두 방법의 통계량 비교 ===')
print(f'방법 1 평균: {np.mean(method1):.4f}  |  방법 2 평균 (역변환): {np.mean(method2_heads):.4f}')
print(f'방법 1 분산: {np.var(method1):.4f}  |  방법 2 분산 (역변환): {np.var(method2_heads):.4f}')
print(f'이론  평균: {10*0.5:.4f}      |  이론  분산: {10*0.5*0.5:.4f}')

### 해석

- 동일한 시드로 생성했기에 두 방법의 결과가 **완벽히 동일**하다.
- **방법 1** (순차적 던지기)과 **방법 2** (동시 던지기)는 수학적으로 동치이다.
- 이유: 독립 시행에서는 **순서가 결과에 영향을 미치지 않기** 때문이다.
- $H=+1, T=-1$ 변환은 단순한 **선형 변환** ($\text{sum} = 2k - n$)이므로 분포의 형태를 바꾸지 않는다.

---
## 5. n이 커지면 정규분포에 수렴 (n=10, 50, 100, 500)

In [ ]:
# === n이 커질 때 이항분포 → 정규분포 수렴 ===
n_values = [10, 50, 100, 500]
p = 0.5
n_samples = 10000

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for idx, n_val in enumerate(n_values):
    # 시뮬레이션
    sim_data = binom.rvs(n_val, p, size=n_samples)
    
    # 이론적 정규분포
    mu = n_val * p
    sigma = np.sqrt(n_val * p * (1 - p))
    
    # 히스토그램
    axes[idx].hist(sim_data, bins=min(n_val + 1, 50), density=True,
                   color=sns.color_palette('Set2')[idx], edgecolor='gray',
                   alpha=0.7, label=f'시뮬레이션 B({n_val}, {p})')
    
    # 정규분포 곡선 오버레이
    x_norm = np.linspace(mu - 4*sigma, mu + 4*sigma, 300)
    axes[idx].plot(x_norm, norm.pdf(x_norm, mu, sigma), 'r-',
                   linewidth=2.5, label=f'N({mu:.0f}, {sigma:.2f}²)')
    
    axes[idx].set_title(f'n = {n_val}', fontsize=15, fontweight='bold')
    axes[idx].set_xlabel('성공 횟수', fontsize=12)
    axes[idx].set_ylabel('밀도', fontsize=12)
    axes[idx].legend(fontsize=10)
    
    # 기댓값/표준편차 텍스트
    axes[idx].text(0.98, 0.95, f'$\mu$ = {mu:.0f}\n$\sigma$ = {sigma:.2f}',
                   transform=axes[idx].transAxes, fontsize=11,
                   verticalalignment='top', horizontalalignment='right',
                   bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('이항분포 → 정규분포 수렴 (중심극한정리)', fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 해석

- **$n = 10$**: 이항분포의 이산적 특성이 뚜렷하게 보이며, 정규분포 곡선과 약간의 차이가 있다.
- **$n = 50$**: 히스토그램이 정규분포 곡선에 매우 잘 들어맞기 시작한다.
- **$n = 100$**: 사실상 정규분포와 구별하기 어려운 수준.
- **$n = 500$**: 완벽한 정규분포 형태. 이산적 특성이 전혀 보이지 않는다.

> **중심극한정리(Central Limit Theorem)**: $n \to \infty$이면 $\frac{X - np}{\sqrt{np(1-p)}} \xrightarrow{d} N(0, 1)$

Galton Board에서 핀의 단수(n)가 많아질수록, 구슬이 쌓이는 패턴이 정규분포에 가까워지는 것과 동일한 원리이다.

---
## 6. Galton Board 애니메이션: 단계별 구슬 낙하 (Static Frames)

In [ ]:
# === Galton Board: 단계별 구슬 낙하 과정 (Static Frames) ===
def simulate_galton_paths(n_levels, n_balls):
    """각 구슬의 경로를 시뮬레이션"""
    paths = []
    for _ in range(n_balls):
        x = 0  # 시작 위치 (중앙)
        path = [x]
        for level in range(n_levels):
            # 각 핀에서 왼쪽(-0.5) 또는 오른쪽(+0.5)
            step = np.random.choice([-0.5, 0.5])
            x += step
            path.append(x)
        paths.append(path)
    return paths

n_levels = 10
n_balls = 200
paths = simulate_galton_paths(n_levels, n_balls)

# 최종 위치 (bin)
final_positions = [p[-1] for p in paths]

# 4단계 스냅샷: 구슬 50개, 100개, 150개, 200개가 떨어진 상태
snapshots = [50, 100, 150, 200]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
axes = axes.ravel()

for idx, n_show in enumerate(snapshots):
    ax = axes[idx]
    
    # 핀 배열 그리기
    for level in range(n_levels):
        n_pins = level + 1
        for pin in range(n_pins):
            pin_x = -level/2 + pin
            ax.plot(pin_x, -(level + 0.5), 'k^', markersize=4, alpha=0.3)
    
    # 구슬 경로 (반투명 선)
    for i in range(min(n_show, 20)):  # 처음 20개만 경로 표시
        path = paths[i]
        levels = range(len(path))
        ax.plot(path, [-l for l in levels], '-', alpha=0.15,
                color=sns.color_palette('Set2')[idx], linewidth=0.8)
    
    # 바닥에 쌓인 구슬 (히스토그램)
    current_finals = [paths[i][-1] for i in range(n_show)]
    bins = np.arange(-n_levels/2 - 0.5, n_levels/2 + 1.5, 1)
    counts, edges = np.histogram(current_finals, bins=bins)
    bin_centers = (edges[:-1] + edges[1:]) / 2
    
    # 쌓인 구슬을 아래에 표현
    bottom_y = -(n_levels + 1)
    max_count = max(counts) if max(counts) > 0 else 1
    for j, (center, count) in enumerate(zip(bin_centers, counts)):
        for ball_idx in range(count):
            ax.plot(center, bottom_y - ball_idx * 0.3, 'o',
                    color=sns.color_palette('Set2')[idx],
                    markersize=4, alpha=0.6)
    
    ax.set_xlim(-6.5, 6.5)
    ax.set_ylim(bottom_y - max_count * 0.3 - 1, 1)
    ax.set_title(f'구슬 {n_show}개 낙하 완료', fontsize=14, fontweight='bold')
    ax.set_xlabel('위치', fontsize=12)
    ax.set_ylabel('단계', fontsize=12)
    ax.axhline(y=bottom_y + 0.5, color='brown', linewidth=2, linestyle='-', alpha=0.5)

plt.suptitle('Galton Board: 단계별 구슬 낙하 과정', fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# === 최종 결과: Galton Board 구슬 분포 vs 정규분포 ===
fig, ax = plt.subplots(figsize=(14, 7))

# 최종 위치 히스토그램
ax.hist(final_positions, bins=np.arange(-5.5, 6.5, 1), density=True,
        color=sns.color_palette('Set2')[0], edgecolor='black',
        alpha=0.7, label=f'Galton Board ({n_balls}개 구슬)')

# 이항분포 PMF (오른쪽으로 간 횟수 스케일)
k_vals = np.arange(0, n_levels + 1)
positions = k_vals - n_levels / 2  # 중앙 기준 위치
ax.plot(positions, binom.pmf(k_vals, n_levels, 0.5), 'rs-', markersize=10,
        linewidth=2, label=f'이론 B({n_levels}, 0.5)', zorder=5)

# 정규분포 곡선
mu = 0
sigma = np.sqrt(n_levels * 0.25)
x_norm = np.linspace(-6, 6, 300)
ax.plot(x_norm, norm.pdf(x_norm, mu, sigma), 'b--', linewidth=2.5,
        label=f'정규분포 N(0, {sigma:.2f}²)', alpha=0.8)

ax.set_xlabel('최종 위치 (중앙 = 0)', fontsize=14)
ax.set_ylabel('밀도', fontsize=14)
ax.set_title(f'Galton Board 최종 결과 ({n_levels}단계, {n_balls}개 구슬)',
             fontsize=16, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

### 해석

- Galton Board 시뮬레이션의 최종 분포가 **이항분포(빨간 점)** 및 **정규분포(파란 선)**와 모두 잘 일치한다.
- 이것이 바로 Galton Board의 핵심 교훈: **단순한 이진 선택(좌/우)의 반복이 종형 곡선을 만들어낸다.**
- 구슬의 수가 많을수록 정규분포에 더 가까워진다.

---
## 7. 정리

| 시뮬레이션 | 방법 | 결과 분포 | 핵심 |
|-----------|------|-----------|------|
| 1 | 동전 1개 × 10번 × 1490회 | $B(10, 0.5)$ | 앞면 개수의 빈도 분포 |
| 2 | 동전 10개 × 1번 × 149회 | $B(10, 0.5)$ (스케일 변환) | H=+1, T=-1 합 - 5 |
| Galton Board | 10단계 핀 통과 | 이항 → 정규 | 물리적 시각화 |

**핵심 요약**:
1. **Galton Board**는 베르누이 → 이항 → 정규분포를 물리적으로 보여주는 장치이다.
2. "1개를 $n$번" 던지기와 "$n$개를 1번" 던지기는 **수학적으로 동치**이다.
3. $H=+1, T=-1$ 변환은 **선형 변환**이므로 분포의 본질을 바꾸지 않는다.
4. $n$이 커질수록 이항분포는 **정규분포에 수렴**한다 (중심극한정리).